In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import torch
import torchvision.transforms as transforms
from constants import TASK2_DATA
from pytorch_pipeline import ImageDataset, create_unfrozen_resnet, train_resnet_model
from torch.utils.data import DataLoader

metadata = pd.read_csv(f"./datasets/{TASK2_DATA}/train_metadata.csv")

# encode bird labels to integers 0-9
le = LabelEncoder()
metadata["encoded_label"] = le.fit_transform(metadata["class_name"])

# split
train_df, val_df = train_test_split(
    metadata, test_size=0.2, random_state=2718, stratify=metadata["encoded_label"]
)

# image preprocessing that ImageNet requires
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# build data loaders
train_dataset = ImageDataset(
    train_df, base_dir=f"./datasets/{TASK2_DATA}/", transform=preprocess
)
val_dataset = ImageDataset(
    val_df, base_dir=f"./datasets/{TASK2_DATA}/", transform=preprocess
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# initialise model
model = create_unfrozen_resnet(num_classes=10)

best_model = train_resnet_model(
    model, train_loader=train_loader, val_loader=val_loader, epochs=10
)

torch.save(best_model.state_dict(), "finetuned_resnet_weights.pth")

Started training on cpu...

Epoch 1/10
---------------
Inputs type: <class 'torch.Tensor'> | Labels type: <class 'torch.Tensor'>
Inputs type: <class 'torch.Tensor'> | Labels type: <class 'torch.Tensor'>
Inputs type: <class 'torch.Tensor'> | Labels type: <class 'torch.Tensor'>
Inputs type: <class 'torch.Tensor'> | Labels type: <class 'torch.Tensor'>


KeyboardInterrupt: 